<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/00_fundamentos/01_pipelines_reproducibilidad.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Pipelines y reproducibilidad

**Pregunta guía:** ¿Qué hace repetible y auditable un experimento?<br>
**Duración sugerida:** 2 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Objetivos

1. Diferenciar aleatoriedad controlada de determinismo absoluto.
2. Encapsular imputación, escala y modelo en un `Pipeline`.
3. registrar configuración, versiones y resultados sin copiar celdas;
4. guardar y recuperar el pipeline completo.

Un resultado reproducible especifica datos, código, dependencias,
semilla, partición y métrica. Si $\hat\mu$ y $\hat\sigma$ se calculan
con entrenamiento, una medida nueva se transforma como
$z=(x-\hat\mu)/\hat\sigma$ usando **esos mismos parámetros**.


In [ ]:
import json
import platform
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
print({"python": platform.python_version(), "sklearn": sklearn.__version__})


## Un experimento pequeño

Simulamos cinco canales de un detector y una energía objetivo. Insertamos
valores ausentes sólo para mostrar que la imputación también debe estar
dentro del pipeline.


In [ ]:
X_array, y_array = make_regression(
    n_samples=500,
    n_features=5,
    n_informative=4,
    noise=12.0,
    random_state=SEMILLA,
)
X = pd.DataFrame(X_array, columns=[f"canal_{i}" for i in range(5)])
y = pd.Series(y_array, name="energía")
mascara = rng.random(X.shape) < 0.03
X = X.mask(mascara)

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA
)
pipe = Pipeline(
    [
        ("imputación", SimpleImputer(strategy="median")),
        ("escala", StandardScaler()),
        ("modelo", Ridge()),
    ]
)
cv = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)
búsqueda = GridSearchCV(
    pipe,
    {"modelo__alpha": np.logspace(-3, 3, 13)},
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
).fit(X_dev, y_dev)

pred = búsqueda.best_estimator_.predict(X_test)
resultado = {
    "semilla": SEMILLA,
    "mejor_alpha": búsqueda.best_params_["modelo__alpha"],
    "rmse_test": root_mean_squared_error(y_test, pred),
    "mae_test": mean_absolute_error(y_test, pred),
}
print(json.dumps(resultado, indent=2))


## Persistir el objeto correcto

Guardar sólo el estimador perdería la mediana y la escala aprendidas.
Guardamos el pipeline entero y comprobamos que las predicciones sean
idénticas. El archivo temporal se elimina automáticamente.


In [ ]:
with tempfile.TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "pipeline.joblib"
    joblib.dump(búsqueda.best_estimator_, ruta)
    recuperado = joblib.load(ruta)
    diferencia = np.max(np.abs(pred - recuperado.predict(X_test)))

print(f"Diferencia máxima después de recuperar: {diferencia:.2e}")
assert diferencia == 0.0


## Lista de control y ejercicios

- ¿El test se separó antes de seleccionar hiperparámetros?
- ¿Todas las transformaciones que aprenden están en el pipeline?
- ¿Se registran versiones, semilla, configuración y métrica?
- ¿El artefacto guardado recibe datos crudos en el mismo esquema?

**Ejercicios:** cambie la imputación por media; repita con cinco semillas;
guarde en JSON media y desviación de RMSE; explique qué parte sigue sin
ser determinista si se usa una GPU.
